# AI-Based Fake Job Posting Detection System
## Part 2: Model Preprocessing, Training and Evaluation

This notebook covers the text cleaning, pipeline creation, training splits, model fitting, and metrics comparison for Naive Bayes, Logistic Regression, and Random Forest Classifiers.

In [ ]:
import re
import os
import joblib
import pandas as pd
import numpy as np

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, f1_score, confusion_matrix, roc_auc_score

### 1. Define NLP cleaning pipeline
We setup stopword lists and wordnet lemmatizers to clean the job posting text fields.

In [ ]:
lemmatizer = WordNetLemmatizer()
try:
    stop_words = set(stopwords.words('english'))
except Exception:
    nltk.download('stopwords')
    stop_words = set(stopwords.words('english'))

def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r'<[^>]*>', ' ', text)  # remove HTML tags
    text = text.lower()                   # lowercase
    text = re.sub(r'[^a-zA-Z\s]', ' ', text) # keep letters and spaces
    try:
        tokens = word_tokenize(text)
    except Exception:
        nltk.download('punkt')
        nltk.download('punkt_tab')
        tokens = word_tokenize(text)
    cleaned = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words and len(word) > 2]
    return " ".join(cleaned)

### 2. Load & Prepare Text Data
We fill null values and concatenate job Title, Company Profile, Description, Requirements, and Benefits into a single textual representation.

In [ ]:
data_path = "../data/fake_job_postings.csv"
df = pd.read_csv(data_path)

print("Combining text fields...")
df['combined_text'] = (
    df['title'].fillna('') + " " +
    df['company_profile'].fillna('') + " " +
    df['description'].fillna('') + " " +
    df['requirements'].fillna('') + " " +
    df['benefits'].fillna('')
)

print("Cleaning combined text (this takes about 1-2 minutes)...")
df['cleaned_text'] = df['combined_text'].apply(clean_text)
print("Cleaning completed! Samples:")
df[['title', 'cleaned_text']].head()

### 3. Split Dataset
We split the dataset into stratified training (80%) and testing splits (20%) using the target variable `fraudulent`.

In [ ]:
X = df[['cleaned_text', 'telecommuting', 'has_company_logo', 'has_questions']]
y = df['fraudulent']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Train set size: {len(X_train)}")
print(f"Test set size: {len(X_test)}")
print(f"Fraud rate in train set: {y_train.mean():.4f}")

### 4. Create Column Preprocessor Pipeline
We setup a ColumnTransformer that vectorizes the cleaned text column using TF-IDF (extracting unigrams and bigrams up to 10,000 features) and lets the binary metadata flags pass through unaltered.

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('text', TfidfVectorizer(max_features=10000, ngram_range=(1,2)), 'cleaned_text'),
        ('meta', 'passthrough', ['telecommuting', 'has_company_logo', 'has_questions'])
    ]
)

### 5. Train and Benchmark Classifiers
We train Naive Bayes (a classic baseline), Logistic Regression (interpretable), and Random Forest (ensemble) to evaluate and compare performance metrics.

In [ ]:
models = {
    'Naive Bayes': MultinomialNB(),
    'Logistic Regression (Balanced)': LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42),
    'Random Forest (Balanced)': RandomForestClassifier(class_weight='balanced', n_estimators=100, random_state=42, n_jobs=-1)
}

for name, model in models.items():
    print(f"\n{'='*15} {name} {'='*15}")
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', model)
    ])
    
    # Train
    pipeline.fit(X_train, y_train)
    
    # Predict
    y_pred = pipeline.predict(X_test)
    
    # Evaluate
    acc = accuracy_score(y_test, y_pred)
    f1_fraud = f1_score(y_test, y_pred, pos_label=1)
    
    print(f"Overall Accuracy: {acc * 100:.2f}%")
    print(f"Fraudulent F1-Score: {f1_fraud:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=["Genuine", "Fraudulent"]))
    
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))

### 6. Export Best Production Pipeline
Typically, Random Forest achieves the highest Fraudulent F1-score (~0.72) and overall accuracy (~97.9%), making it the best production candidate. We export the trained model pipeline.

In [ ]:
best_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(class_weight='balanced', n_estimators=100, random_state=42, n_jobs=-1))
])

print("Fitting best model on full training set...")
best_pipeline.fit(X_train, y_train)

output_model_path = "../src/model.joblib"
print(f"Exporting model to {output_model_path}...")
joblib.dump(best_pipeline, output_model_path)
print("Model saved successfully!")